# Time-Series Sequence Preparation

## Objective

Prepare the leakage-safe time-series dataset for classical machine learning
and deep learning experiments.


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(
    r"C:\Users\sun\OneDrive\Desktop\Northgate_AI_Stock_Predictor")

DATA_DIR = PROJECT_ROOT / "data"
MODEL_DATA_DIR = DATA_DIR / "processed"
RESULTS_DIR = PROJECT_ROOT / "results"

INPUT_FILE = MODEL_DATA_DIR / "aapl_features_with_target.parquet"

print("Input file:", INPUT_FILE)
print("Exists:", INPUT_FILE.exists())

Input file: C:\Users\sun\OneDrive\Desktop\Northgate_AI_Stock_Predictor\data\processed\aapl_features_with_target.parquet
Exists: True


In [2]:
df = pd.read_parquet(INPUT_FILE)

df["date"] = pd.to_datetime(df["date"])
df = df.sort_values("date").reset_index(drop=True)

print("Shape:", df.shape)
print("Date range:", df["date"].min(), "→", df["date"].max())

display(df.head())

Shape: (2931, 291)
Date range: 2015-01-02 00:00:00 → 2026-08-28 00:00:00


,date,Adj Close_AAPL,Adj Close_CAT,Adj Close_HD,Adj Close_JNJ,Adj Close_JPM,Adj Close_KO,Adj Close_MSFT,Adj Close_NVDA,Adj Close_PG,...,Features_day_of_week,Features_month,Features_quarter,Features_day_of_month,Features_month_sin,Features_month_cos,Features_day_of_week_sin,Features_day_of_week_cos,target,future_return_5d
0,2015-01-02,24.171755,68.899345,78.204041,75.737366,46.066788,29.390263,39.607178,0.482423,65.069992,...,4,1,1,2,0.5,0.866025,-0.951057,0.309017,1,0.024513
1,2015-01-05,23.490805,65.262398,76.563309,75.208389,44.636635,29.390263,39.242962,0.474275,64.760643,...,0,1,1,5,0.5,0.866025,0.000000,1.000000,1,0.028235
2,2015-01-06,23.493017,64.842461,76.328911,74.838837,43.479263,29.613441,38.666973,0.459895,64.465637,...,1,1,1,6,0.5,0.866025,0.951057,0.309017,1,0.037267
3,2015-01-07,23.822432,65.847313,78.945038,76.490990,43.545609,29.983099,39.158257,0.458697,64.803787,...,2,1,1,7,0.5,0.866025,0.587785,-0.809017,1,0.019026
4,2015-01-08,24.737753,66.522217,80.691635,77.092400,44.518696,30.345751,40.310219,0.475952,65.544853,...,3,1,1,8,0.5,0.866025,-0.587785,-0.809017,0,-0.045312


In [3]:
# CHRONOLOGICAL TRAIN / VALIDATION / TEST SPLIT

n = len(df)

train_end = int(n * 0.70)
val_end = int(n * 0.85)

train_df = df.iloc[:train_end].copy()
val_df = df.iloc[train_end:val_end].copy()
test_df = df.iloc[val_end:].copy()

print("Total samples :", len(df))
print("Train samples :", len(train_df))
print("Validation    :", len(val_df))
print("Test samples  :", len(test_df))

print("\nDate ranges")
print()

Total samples : 2931
Train samples : 2051
Validation    : 440
Test samples  : 440

Date ranges



In [4]:
print(
    "Train:",
    train_df["date"].min(),
    "→",
    train_df["date"].max())

Train: 2015-01-02 00:00:00 → 2023-02-24 00:00:00


In [5]:

print(
    "Validation:",
    val_df["date"].min(),
    "→",
    val_df["date"].max())

Validation: 2023-02-27 00:00:00 → 2024-11-22 00:00:00


In [6]:
print(
    "Test:",
    test_df["date"].min(),
    "→",
    test_df["date"].max())

Test: 2024-11-25 00:00:00 → 2026-08-28 00:00:00


In [7]:
# LEAKAGE CONTROL

TARGET_COLUMN = "target"

LEAKAGE_COLUMNS = [
    "target",
    "future_return_5d"]

print("Target:", TARGET_COLUMN)
print("Excluded leakage columns:", LEAKAGE_COLUMNS)

Target: target
Excluded leakage columns: ['target', 'future_return_5d']


In [8]:
# FEATURE MATRIX

EXCLUDE_COLUMNS = [
    "date",
    "target",
    "future_return_5d"]

FEATURE_COLUMNS = [
    column
    for column in df.columns
    if column not in EXCLUDE_COLUMNS]

print("Total features:", len(FEATURE_COLUMNS))

print("\nFirst 30 features:")
for feature in FEATURE_COLUMNS[:30]:
    print(feature)

Total features: 288

First 30 features:
Adj Close_AAPL
Adj Close_CAT
Adj Close_HD
Adj Close_JNJ
Adj Close_JPM
Adj Close_KO
Adj Close_MSFT
Adj Close_NVDA
Adj Close_PG
Adj Close_XOM
Adj Close_^GSPC
Adj Close_^VIX
Close_AAPL
Close_CAT
Close_HD
Close_JNJ
Close_JPM
Close_KO
Close_MSFT
Close_NVDA
Close_PG
Close_XOM
Close_^GSPC
Close_^VIX
High_AAPL
High_CAT
High_HD
High_JNJ
High_JPM
High_KO


In [9]:
# FEATURE LEAKAGE AUDIT

suspicious_keywords = [
    "future",
    "target",
    "forward",
    "next",
    "lead"]

suspicious_features = [
    feature
    for feature in FEATURE_COLUMNS
    if any(
        keyword in feature.lower()
        for keyword in suspicious_keywords)]

print("Suspicious features found:", len(suspicious_features))

for feature in suspicious_features:
    print(feature)

Suspicious features found: 0


In [10]:
# NUMERIC FEATURE MATRIX

numeric_features = [
    feature
    for feature in FEATURE_COLUMNS
    if pd.api.types.is_numeric_dtype(df[feature])]

print("Numeric features:", len(numeric_features))

Numeric features: 288


In [11]:
# MISSING VALUE AUDIT BY SPLIT

for name, dataset in [
    ("TRAIN", train_df),
    ("VALIDATION", val_df),
    ("TEST", test_df)]:
    
    missing = dataset[numeric_features].isna().sum().sum()
    total = dataset[numeric_features].size
    
    print(
        f"{name}: "
        f"{missing:,} missing values "
        f"({missing / total * 100:.2f}%)")

TRAIN: 7,932 missing values (1.34%)
VALIDATION: 0 missing values (0.00%)
TEST: 0 missing values (0.00%)


In [12]:
# TRAIN-ONLY STANDARDIZATION

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(
    train_df[numeric_features])

X_val = scaler.transform(
    val_df[numeric_features])

X_test = scaler.transform(
    test_df[numeric_features])


y_train = train_df[TARGET_COLUMN].values
y_val = val_df[TARGET_COLUMN].values
y_test = test_df[TARGET_COLUMN].values

print("X_train:", X_train.shape)
print("X_val  :", X_val.shape)
print("X_test :", X_test.shape)

print("\ny_train:", y_train.shape)
print("y_val  :", y_val.shape)
print("y_test :", y_test.shape)

X_train: (2051, 288)
X_val  : (440, 288)
X_test : (440, 288)

y_train: (2051,)
y_val  : (440,)
y_test : (440,)


In [13]:
# SEQUENCE WINDOW CREATION

LOOKBACK = 60

def create_sequences(X, y, dates, lookback=60):
    X_seq = []
    y_seq = []
    date_seq = []

    for i in range(lookback, len(X)):
        X_seq.append(
            X[i - lookback:i])
        
        y_seq.append(
            y[i])
        
        date_seq.append(
            dates.iloc[i])

    return (
        np.array(X_seq),
        np.array(y_seq),
        pd.Series(date_seq))

In [14]:
# BUILD SEQUENCES

X_train_seq, y_train_seq, train_dates = create_sequences(
    X_train,
    y_train,
    train_df["date"].reset_index(drop=True),
    LOOKBACK)

X_val_seq, y_val_seq, val_dates = create_sequences(
    X_val,
    y_val,
    val_df["date"].reset_index(drop=True),
    LOOKBACK)

X_test_seq, y_test_seq, test_dates = create_sequences(
    X_test,
    y_test,
    test_df["date"].reset_index(drop=True),
    LOOKBACK)

In [15]:
print("Train sequence:", X_train_seq.shape)
print("Validation sequence:", X_val_seq.shape)
print("Test sequence:", X_test_seq.shape)

Train sequence: (1991, 60, 288)
Validation sequence: (380, 60, 288)
Test sequence: (380, 60, 288)


In [17]:
# SEQUENCE SANITY CHECK

print("Lookback window:", LOOKBACK)

print(
    "\nFirst training sequence:",
    train_dates.iloc[0])

print(
    "Last training sequence:",
    train_dates.iloc[-1])

print(
    "\nFirst test sequence:",
    test_dates.iloc[0])

print(
    "Last test sequence:",
    test_dates.iloc[-1])

Lookback window: 60

First training sequence: 2015-03-31 00:00:00
Last training sequence: 2023-02-24 00:00:00

First test sequence: 2025-02-25 00:00:00
Last test sequence: 2026-08-28 00:00:00


In [18]:
print("\nTarget classes:")
print(
    "Train:", np.unique(y_train_seq, return_counts=True))

print(
    "Validation:", np.unique(y_val_seq, return_counts=True))

print(
    "Test:", np.unique(y_test_seq, return_counts=True))


Target classes:
Train: (array([0, 1]), array([ 848, 1143]))
Validation: (array([0, 1]), array([164, 216]))
Test: (array([0, 1]), array([170, 210]))


In [19]:
# SAVE PREPARED SEQUENCES

SEQUENCE_DIR = RESULTS_DIR / "sequence_data"
SEQUENCE_DIR.mkdir(parents=True, exist_ok=True)

np.save(SEQUENCE_DIR / "X_train_seq.npy", X_train_seq)
np.save(SEQUENCE_DIR / "y_train_seq.npy", y_train_seq)

np.save(SEQUENCE_DIR / "X_val_seq.npy", X_val_seq)
np.save(SEQUENCE_DIR / "y_val_seq.npy", y_val_seq)

np.save(SEQUENCE_DIR / "X_test_seq.npy", X_test_seq)
np.save(SEQUENCE_DIR / "y_test_seq.npy", y_test_seq)

train_dates.to_csv(
    SEQUENCE_DIR / "train_dates.csv",
    index=False)

val_dates.to_csv(
    SEQUENCE_DIR / "val_dates.csv",
    index=False)

test_dates.to_csv(
    SEQUENCE_DIR / "test_dates.csv",
    index=False)

In [20]:
print("Sequence datasets saved successfully")
print("Location:", SEQUENCE_DIR)

Sequence datasets saved successfully
Location: C:\Users\sun\OneDrive\Desktop\Northgate_AI_Stock_Predictor\results\sequence_data
